# 📓 Diário de Bordo: Radar Mercado Dados SP
## Sprint 1: Épico [INFRA] - Configuração de Ambiente

**Objetivo:** Provisionar e validar a infraestrutura base do projeto utilizando Docker, garantindo um ambiente isolado para o banco de dados e para a orquestração.

### 1. Arquitetura Escolhida (Task-2)
Para otimizar os recursos locais da máquina, optamos por uma arquitetura enxuta com dois serviços principais:
* 1.Um container isolado para o PostgreSQL, que será a nossa Landing Zone (onde os dados do projeto vão ficar). 
* 2.Um container do Airflow rodando em modo standalone, que inicializa todos os serviços necessários dele internamente sem consumir toda a sua máquina. 

### 2. Análise do `docker-compose.yml`
Abaixo estão as minhas anotações detalhadas sobre a função de cada linha crítica na criação da nossa infraestrutura:

**Sobre o PostgreSQL (Landing Zone):**
* `image: postgres:15`: A instrução principal.Diz ao Docker para ir até a internet (Docker Hub) e baixar a imagem oficial do banco de dados PostgreSQL, especificamente a versão 15. 
 `volumes:` -> `- postgres_data:/var/lib/postgresql/data`:Ela mapeia uma pasta virtual (postgres_data) para a pasta interna do container onde o Postgres salva as tabelas.Se você desligar o container, os dados ficam a salvo no seu PC. 

**Sobre o Apache Airflow (Orquestrador):**
* `image: apache/airflow:2.8.1`: Baixa a imagem oficial do Apache Airflow (versão 2.8.1, bem recente e estável). 
 * `- AIRFLOW__CORE__LOAD_EXAMPLES=False`: O Airflow vem com dezenas de DAGs de "tutorial" que poluem a tela.Essa linha diz para ele não carregar esse lixo, deixando a interface limpa apenas para o seu projeto. 
 * `- ./dags:/opt/airflow/dags`: Pega a pasta /dags que você criou hoje e a "espelha" para dentro do container.Qualquer código Python que você salvar no seu VS Code aparecerá automaticamente no Airflow. 

 * `- ./scripts:/opt/airflow/scripts`: Faz o mesmo espelhamento para a pasta de scripts. 

* `command: standalone`: O pulo do gato.Em vez de subir 6 containers pesados que o Airflow exige por padrão, esse comando roda todos os serviços internos do Airflow (webserver, scheduler, metadata) em um único processo, economizando muita memória RAM do seu PC. 
* `depends_on: - postgres_landing`: Uma regra de segurança.Diz ao Docker: "Só ligue o Airflow depois que o container do banco de dados já estiver 100% ligado". 

**Sobre Redes (Networking):**
* `driver: bridge`: O tipo de rede. O "bridge" (ponte) permite que containers no mesmo computador conversem entre si pelo nome (ou seja, o Airflow vai conseguir chamar o Postgres pelo nome, sem precisar saber o IP dele). 

### 3. Integração e Acessos (Task-3)
 * **Configuração da Connection no Airflow:** Para conectar o orquestrador ao banco de dados pela interface web, a configuração do `Host` exige atenção especial: `Host: postgres_landing_zone` (Aqui está o pulo do gato! Como eles estão na mesma rede Docker, você não usa 'localhost', você usa o nome exato do container que definimos no YAML). 

### 4. Validação de Persistência e Resiliência (Task-4)
O conceito de volumes garante que, mesmo que a infraestrutura seja destruída, os dados permaneçam intactos. O fluxo abaixo comprova essa persistência.

In [ ]:
# 1. criar uma tabela e fazer um insert
docker exec -it postgres_landing_zone psql -U admin -d radar_sp -c "CREATE TABLE teste_volume (id SERIAL, mensagem VARCHAR(100)); INSERT INTO teste_volume (mensagem) VALUES ('Se eu sobreviver, o volume funciona!');" [cite: 15]

# 2. desligar o Docker
docker-compose down 

# 3. religar o docker
docker-compose up 

# 4. lançar comando de consulta na tabela criada anteriormente
docker exec -it postgres_landing_zone psql -U admin -d radar_sp -c "SELECT * FROM teste_volume;" [cite: 16]

## Sprint 2: Épico [INGEST] - Pipeline de Coleta de Dados

**Objetivo:** Criar os conectores e rotinas em Python para consumir dados de vagas de emprego via API pública, garantindo que as extrações sejam estáveis e seguras antes de enviá-las para o banco de dados.

### 1. Estrutura do Script de Extração via API (Task-5)
Optamos por utilizar a **API da Adzuna** (um agregador de vagas) em vez de fazer *web scraping* direto em sites como LinkedIn ou Gupy. Isso evita que o pipeline quebre por mudanças estruturais de HTML ou bloqueios de robôs.

Abaixo está o detalhamento lógico do script `extracao_adzuna.py`:

**1. O Cinto de Utilidades (Bibliotecas):**
* `os`: Biblioteca nativa para ler variáveis de ambiente do sistema.
* `requests`: O nosso "carteiro", responsável por fazer as requisições HTTP (GET) na internet.
* `pandas`: Transforma o arquivo JSON complexo retornado pela API em uma tabela estruturada (DataFrame).
* `dotenv` (`load_dotenv`): O "chaveiro" que acessa o arquivo oculto `.env` para proteger nossas chaves de API.

**2. O Cofre de Segurança:**
O uso do `load_dotenv()` garante que o `APP_ID` e o `APP_KEY` não fiquem expostos no código-fonte, evitando vazamento de credenciais ao enviar o projeto para o GitHub.

**3. Passagem de Parâmetros Segura:**
Em vez de concatenar a URL manualmente (o que frequentemente causa erros `400` por conta de espaços em branco), passamos os filtros (cargo, local, etc.) em um dicionário Python. A biblioteca `requests` formata isso de forma segura.

**4. Controle de Qualidade (Status Code):**
* `200 (OK)`: Requisição aceita e dados recebidos.
* `403 (Forbidden)`: Acesso negado, indicando erro nas credenciais do `.env`.

**5. Transformação (Achatamento do JSON):**
A função `pd.json_normalize()` é utilizada para converter a lista de resultados aninhados do JSON em uma estrutura tabular de linhas e colunas perfeita para a futura ingestão no banco de dados.

# Código base utilizado no arquivo /scripts/extracao_adzuna.py

In [ ]:

import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
APP_ID = os.getenv('ADZUNA_APP_ID')
APP_KEY = os.getenv('ADZUNA_APP_KEY')

def buscar_vagas_sp():
    url = "https://api.adzuna.com/v1/api/jobs/br/search/1"
    
    parametros = {
        "app_id": APP_ID,
        "app_key": APP_KEY,
        "results_per_page": 10,
        "what": "Data Engineer",
        "where": "São Paulo",
        "content-type": "application/json"
    }

    resposta = requests.get(url, params=parametros)

    if resposta.status_code == 200:
        dados = resposta.json()
        vagas = dados.get('results', [])
        df = pd.json_normalize(vagas)
        
        if not df.empty:
            print("Vagas extraídas com sucesso!")
            print(df[['title', 'company.display_name', 'location.display_name']].head())
    else:
        print(f"Erro na requisição. Código: {resposta.status_code}")

if __name__ == "__main__":
    buscar_vagas_sp()

### 2. Contexto de Execução e Validação de Ingestão (Task-5)

**Dúvida de Arquitetura:** Ao executar o script de extração localmente e salvar no banco, é possível visualizar o log de sucesso dessa ação na interface do Airflow?

**Análise e Resposta:** Não. O log não aparecerá no orquestrador devido à segregação de ambientes do Docker:
* **Quem executou (A Origem):** Quando rodamos `python scripts/extracao_adzuna.py`, a execução ocorreu através do interpretador Python físico do Windows (o ambiente `.venv`). O Airflow, que está isolado dentro de um container Linux, só gera logs de processos que ele mesmo dispara (as suas próprias DAGs).
* **O Destino Compartilhado:** Apesar do Airflow não ter "visto" o script rodar, o dado chegou ao destino correto. O script apontou para `localhost:5432`, e o serviço de rede do Docker interceptou essa chamada e a direcionou para o container `postgres_landing_zone`.

**Validação Final (A Prova Real):**
Para garantir que a jornada "Extração (API) -> Transformação (Pandas) -> Carga (Postgres)" funcionou perfeitamente via script local, validamos a injeção diretamente acessando o container do banco de dados e contando as linhas inseridas na tabela `vagas_brutas_adzuna`.

* Entra no container do Postgres e executa um SELECT COUNT para confirmar a injeção dos dados via script local.
### O resultado esperado é 10 (referente ao número limite de vagas que extraímos no teste inicial).

In [ ]:
docker exec -it postgres_landing_zone psql -U admin -d radar_sp -c "SELECT COUNT(*) FROM vagas_brutas_adzuna;"

### 3. Estruturação do Banco de Dados e Tipagem (Task-6)

**Objetivo:** Abandonar a criação automática de tabelas do Pandas e criar a tabela oficial `bronze_vagas` com um schema rígido (SQL) para garantir a integridade dos dados na Landing Zone.

**Conceitos e Desafios:**
* **Schema Físico:** Definimos colunas com tipos específicos (`VARCHAR`, `FLOAT`, `TEXT`) e, mais importante, uma restrição `UNIQUE` no `adzuna_id` para evitar duplicatas.
* **O Erro de Tipagem (DatatypeMismatch):** Aprendemos que o PostgreSQL é rigoroso. Quando a API retornava valores nulos para o salário, o Pandas enviava as colunas como texto (string). O banco recusou a entrada, exigindo números.
* **A Solução (Type Casting):** Tratamos os dados no Python usando `pd.to_numeric(..., errors='coerce')` antes da injeção, forçando o formato float e convertendo vazios para `NaN` (que o Postgres aceita perfeitamente como nulo).

### 4. Segurança e Cofre do Airflow (Task-8)

**Objetivo:** Configurar as credenciais de forma segura para que o Airflow, isolado em seu próprio container Linux, consiga acessar a API externa e o Banco de Dados.

**Conceitos e Desafios:**
* **O Problema do Arquivo `.env`:** O Airflow não lê arquivos locais do Windows. A solução foi migrar as chaves para o "Cofre" interno dele.
* **Variables:** Cadastramos `adzuna_app_id` e `adzuna_app_key` pela interface web, garantindo que o código da DAG não contenha senhas expostas (boas práticas de Git).
* **Connections:** Criamos a conexão `postgres_landing_zone_conn`. O detalhe crucial de redes Docker se repetiu: o host não é `localhost`, e sim o nome do container (`postgres_landing_zone`).
* **Troubleshooting de Acesso:** Tivemos um bloqueio com a senha padrão do Airflow standalone. A solução arquitetural foi usar a CLI do Docker para forçar a exclusão do usuário antigo e criar um novo administrador:
  `docker exec -it airflow_orquestrador airflow users create -u admin -f Admin -l User -r Admin -e admin@example.com -p *****`

### 5. A Primeira DAG e a Estratégia de Upsert (Task-7)

**Objetivo:** Automatizar o pipeline para rodar diariamente (`schedule_interval='@daily'`), transformando um script local em uma rotina profissional.

**Conceitos e Desafios:**
* **O "Erro de Sucesso" (UniqueViolation):** Na primeira execução, a DAG quebrou porque tentou inserir vagas que já existiam na tabela, ferindo a regra `UNIQUE`. Isso provou que a blindagem da Task-6 funcionou!
* **A Solução Profissional (Padrão Upsert):** Para lidar com execuções diárias (onde a API traz vagas velhas misturadas com novas), abandonamos o `if_exists='append'` direto do Pandas e implementamos o padrão de mercado:
  1. **Staging:** Injetamos os dados brutos de hoje em uma tabela temporária (`stg_vagas_temp`).
  2. **Merge Seguro:** Usamos SQL nativo (`ON CONFLICT DO NOTHING`) para tentar inserir os dados na `bronze_vagas`.
  3. **Resultado (Idempotência):** Apenas vagas inéditas entram; vagas repetidas são ignoradas silenciosamente. O pipeline agora pode rodar 100 vezes seguidas sem duplicar dados ou quebrar.

### 6. Observabilidade, Logs Estruturados e Resiliência (Task-9)

**Objetivo:** Evoluir o pipeline de um script de execução simples para uma rotina monitorável e resiliente, implementando tratamento de erros (`try/except`) e substituindo saídas de texto padrão por logs estruturados do Apache Airflow.

**A Evolução dos Registros: `print()` vs `logging`**
A principal mudança visual e arquitetural dessa etapa foi na forma como a DAG se comunica conosco pela interface do Airflow:
* **O problema do `print()`:** Em scripts locais, o `print` joga o texto na tela. Mas dentro de um orquestrador rodando em background, esses textos ficam perdidos, sem contexto de tempo ou gravidade.
* **O poder do `logging` nativo:** Ao importar a biblioteca `logging` (`logger.info`, `logger.error`, `logger.warning`), o Airflow passa a interceptar as mensagens e cria uma trilha de auditoria profissional na aba "Logs" da execução.
    * **Rastreabilidade:** Cada linha do log ganha um *timestamp* automático (ex: `[2026-05-01, 22:38:12 UTC]`).
    * **Severidade e Cores:** O log agora entende o que é apenas uma informação do processo (`INFO`), o que é um alerta que não quebra o pipeline (`WARNING`) e o que é uma falha crítica (`ERROR`).

**Tratamento de Erros e o uso do `raise`**
Além dos logs, blindamos as duas áreas de maior risco do pipeline (Rede/API e Banco de Dados) com blocos `try/except`:
1. **Timeouts da API:** Se a Adzuna demorar a responder, o `requests` vai estourar um erro controlado.
2. **Fal

In [ ]:
# Exemplo do Padrão de Projeto implementado na Task-9
import logging
logger = logging.getLogger(__name__)

try:
    logger.info("Iniciando extração da API da Adzuna...")
    # Lógica de extração aqui...
    logger.info("✅ 10 vagas extraídas com sucesso.")

except Exception as e:
    # Registra o erro de forma estruturada no log da DAG antes de falhar a task
    logger.error(f"Erro Crítico: Falha na comunicação com a API - Detalhes: {e}")
    raise # Comunica ao Airflow que a Task falhou (quadrado vermelho)

## Sprint 3: Épico [DATALAKE & PROCESSAMENTO] - Armazenamento de Arquivos e Camada Silver

**Objetivo:** Evoluir a infraestrutura para um padrão de Big Data real, integrando um Object Storage (MinIO) para atuar como Data Lake e utilizando o PySpark (via Jupyter Lab local) para substituir a nuvem do Databricks no processamento dos dados.

### 1. Implementação do Data Lake Local com MinIO (Task-10)

**A Mudança de Paradigma:** Bancos relacionais (PostgreSQL) são ótimos para a Landing Zone, mas custosos para armazenar gigabytes de dados analíticos. A solução foi provisionar um container do **MinIO** (que simula perfeitamente o Amazon S3) para guardar nossos dados brutos (Bronze).

**Desafios de Rede Docker resolvidos:**
* **O Enigma do DNS:** O Airflow não conseguia fazer o upload do arquivo via `boto3` porque a rede do Docker não resolvia o nome `minio-datalake`. 
* **O "Pulo do Gato" (host.docker.internal):** No ecossistema Docker para Windows, quando a rede interna falha, podemos forçar a comunicação a passar pelo "Gateway do Host" usando a URL `http://host.docker.internal:9000`. Isso permitiu que o Airflow saísse do seu container e achasse o MinIO com sucesso.
* **O Jogo das Portas:** Consolidamos o entendimento de que a porta `9000` serve para as APIs e integrações de código (onde o Boto3 e o Spark batem), enquanto a `9001` é exclusiva para a interface web humana.

### 2. A Camada Silver com PySpark (Task-11)

**A Decisão Arquitetural (Local vs Cloud):** Para não depender de processos manuais de upload e manter o pipeline automatizado "End-to-End" na mesma máquina, substituímos o uso do Databricks por um container dedicado do **Jupyter Lab com PySpark**.

**A Lógica de Transformação Aplicada:**
1. **A Ponte S3A:** Configuramos a sessão do Spark com os pacotes `hadoop-aws` para que ele conseguisse ler nativamente do MinIO usando o protocolo `s3a://`.
2. **Qualidade de Dados:** O script faz a leitura do CSV bruto (Bronze) e aplica filtros rigorosos: 
   * `.dropDuplicates(["adzuna_id"])` para remover redundâncias da API.
   * `.dropna()` para excluir vagas inúteis (sem título ou descrição).
   * `trim()` e `lower()` para padronizar os textos.
3. **O Padrão Ouro de Armazenamento (Parquet):** A conversão final de CSV para Parquet é o que garante performance para a Camada Gold. O Parquet é colunar, possui um esquema rígido (sem necessidade de adivinhar tipos) e comprime os arquivos (economizando espaço valioso em arquiteturas de nuvem).

In [ ]:
# Arquitetura base do processamento Silver (PySpark + MinIO)
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, trim

# 1. Conexão com o Data Lake (MinIO)
spark = SparkSession.builder \
    .appName("Transformacao_Silver_Adzuna") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://host.docker.internal:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "admin1234") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

# 2. Leitura da Camada Bronze
df_bronze = spark.read.csv("s3a://radar-sp/bronze/vagas/bronze_vagas.csv", header=True, inferSchema=True)

# 3. Transformações de Qualidade (Silver)
df_deduplicado = df_bronze.dropDuplicates(["adzuna_id"])
df_limpo = df_deduplicado.dropna(subset=["descricao", "titulo"])

df_silver = df_limpo.withColumn("titulo_padronizado", trim(lower(col("titulo")))) \
                    .withColumn("descricao_padronizada", trim(lower(col("descricao"))))

# 4. Escrita Otimizada em Parquet
df_silver.write.mode("overwrite").parquet("s3a://radar-sp/silver/vagas")
print("✅ Camada Silver concluída! Dados limpos e salvos em Parquet.")

### 3. A Lógica de Enriquecimento e Expressões Regulares (Task-12)

Para extrair inteligência de textos longos e não estruturados (as descrições das vagas), implementamos uma lógica de varredura em massa utilizando Regex. Abaixo está o detalhamento técnico do coração desse processo:

```python
df_gold = df_gold.withColumn(
    f"req_{ferramenta}",
    when(col("descricao_padronizada").rlike(padrao_regex), True).otherwise(False)
)

Anatomia do Mecanismo de Extração:

 - df_gold.withColumn(...): Como os DataFrames no PySpark são imutáveis (não podem ser alterados diretamente na memória), esse método instrui o Spark a criar uma nova coluna derivada a cada volta do loop, gerando um novo estado do DataFrame.

- f"req_{ferramenta}": Uma f-string do Python que batiza a nova coluna dinamicamente a cada iteração (ex: na rodada do Python, a coluna se chamará req_python; na rodada do SQL, req_sql).

- when(condição, True).otherwise(False): É o equivalente do PySpark para a estrutura CASE WHEN ... THEN ... ELSE do SQL. Se a condição de busca for atendida, o Spark grava o valor booleano True na célula daquela linha; se não for atendida (otherwise), grava False.

- col("descricao_padronizada").rlike(padrao_regex): O motor de busca real. O método .rlike() significa Regex Like (Expressão Regular). Ele varre o texto limpo da descrição procurando pelo padrão que definimos. O uso dos marcadores de limite de palavra (\b) no padrão garante que o Spark só valide a palavra exata e isolada, blindando a base contra falsos positivos.

# 4.Resumo

### 1. Ingestão (Extract): Airflow consumindo a API da Adzuna de forma resiliente e com logs estruturados.

### 2. Landing Zone (Load): PostgreSQL armazenando os dados em segurança usando a estratégia de Upsert (evitando duplicatas logo na entrada).

### 3.Data Lake (Bronze): Automação com a biblioteca boto3 para descarregar arquivos brutos no MinIO.

### 4.Processamento (Silver): PySpark limpando, padronizando os textos e salvando no formato de alta performance Parquet.

### 5. Enriquecimento Analítico (Gold): O seu "motor de busca" com Regex fatiando textos gigantescos para entregar exatamente quais tecnologias o mercado de São Paulo está exigindo.